# [EDA] 강종별 특성 조사 및 온도 데이터와의 관계 분석

**담당: 정수진 · Issue #34**

**목적:** 같은 소둔로에서도 강종 후보와 LOT에 따라 온도 수준·변동성이 다른지 확인합니다. 강종에 따른 정상 운전 차이를 이상으로 오인하지 않도록 비교 단위를 정하는 분석입니다. 강종의 인과 효과나 고장 판정은 다루지 않습니다.

**핵심 결과:** 이벤트에는 8코드, 온도에는 14 LOT와 7종 후보가 있습니다. TC-Z3의 후보별 행 가중 평균은 약 855.850~860.720, 표준편차는 약 2.061~11.032로 다릅니다. 이 차이를 LOT별·센서별·시간별로 나눠 확인합니다. SPCC는 온도 후보 LOT가 없으며 SGCC·DP590·SPCD는 각각 1 LOT입니다.


<a id="toc"></a>
## 목차

1. [강종 정의와 분석 질문](#definition)
2. [데이터 준비와 LOT–강종 연결](#mapping)
3. [강종·LOT별 온도 통계](#statistics)
4. [시각화로 확인하는 차이](#plots)
5. [결론과 다음 분석자 인계](#handoff)

위에서부터 실행합니다. 결과 표는 전체 센서 통계를 보존하고 본문은 핵심 표와 그림만 표시합니다.


### 최종 결과와 활용 방향

| 얻은 결과 | 구체적인 해석·활용 | 기존 자료로 다음에 볼 것 |
| --- | --- | --- |
| 이벤트 8코드, 온도 14 LOT·7종 후보 | 강종과 LOT는 다른 집계 단위. 후보별 비교 뒤 LOT별로 나눠 해석 | SPCC는 미관측으로 유지; 반복 LOT끼리 비교 |
| TC-Z3 후보별 평균 855.850~860.720, 표준편차 2.061~11.032 | 관측 차이는 있으나 강종 단독 효과는 아님 | 전체 행 평균과 LOT 동일 가중 평균, 센서별 분포 |
| SPHC 내부 LOT 표준편차 1.827~11.303 | 같은 후보에서도 변동성이 크게 다름. 강종 전체에 하나의 정상 범위를 바로 적용하지 않음 | LOT 240061과 다른 SPHC LOT의 시간 변화 비교 |
| LOT 240061 첫/마지막 10분 TC-Z3 평균 820.198 / 857.505 | 후반이 약 37.31 높음. 전체 산포에 초반 상승 추세가 포함됨 | 초반 10분을 5분·15분으로 바꿔 결과 민감도, 다른 존 동시 변화 검토 |
| 같은 LOT 초반 OP-Z3 결측 179/589행(약 30.4%) | 초반 출력 평균은 유효 410행만 반영. 온도–출력 해석은 불완전 | TC·OP가 동시에 있는 시각만 비교, 결측 위치·지속구간 확인 |
| 후보 시작에 DP590→SPHC 이벤트, 전날 CAL01 정비와 계획수리 존재 | 시간상 인접한 조업·정비 맥락을 확인함. 재가동·수리 효과의 증거는 아님 | 아래 실제 사건 대조표와 다른 LOT에서 반복 여부 확인 |
| 강종변경 ±10분 양쪽 관측 0건 | 전환 효과는 계산 불가. 같은 LOT 초반·후반 비교와 구분 | 전후 효과 분석은 제외하고 관측 구간 안의 변화 분석 |

**완료 범위:** 강종 정의·열처리 배경, 후보 연결, 전체 센서 통계, 그래프 3개, 검토 구간의 실제 사건 대조를 수행했습니다. 고장 판정·제어 지연·시간순 모델 검증은 수행하지 않았습니다. 외부 데이터 확보 없이 현재 파일로 가능한 후속 확인만 남깁니다.


**함께 볼 데이터:** LOT 240061의 TC-Z3와 같은 시각의 TC-Z4~Z6·OP-Z3·결측을 먼저 확인하고, 다른 SPHC LOT와 비교합니다. 사건 대조는 CR1 강종변경 → 정기수리캘린더 → CAL01 정비이력 → 해당 WO_NO의 교체 상세 순서입니다. 파일명·컬럼·연결 기준은 5절 하단의 「결과별로 구체적으로 어떤 데이터와 함께 보는가?」 표를 참고합니다.

**왜 이 결과를 중요하게 보는가?** 같은 SPHC 후보 안의 LOT 평균 차이(약 7.60)가 후보별 합산 평균 범위(약 4.87)보다도 큽니다. 따라서 강종별 하나의 기준을 바로 정하기보다 LOT·시간 구간을 나눠 보아야 합니다. 특히 240061은 초반 상승 추세와 출력 결측이 있어 사건 대조 전에 관측 상태를 확인해야 합니다. 이 비교는 기술통계이며 강종보다 다른 요인이 더 크다는 인과 분석은 아닙니다.

**현재 의사결정에 직접 쓰지 않을 것:** 강종 평균 순위를 권장 소둔 온도로 쓰는 것, 큰 표준편차를 고장 점수로 쓰는 것, 이벤트와 가까운 시각을 원인의 증거로 쓰는 것입니다. 해당 값들은 불필요한 것이 아니라 **검토 순서를 정할 단서**이며, 정상·고장 판단을 확정하는 데는 충분하지 않습니다.


<a id="definition"></a>
## 1. 강종 정의와 분석 질문

강종은 성분·기계적 성질·제조 방식·용도에 따른 강재 분류입니다. LOT는 관측한 묶음이므로 강종 수와 같을 필요가 없습니다. 아래는 규격·제조사 자료에 따른 제품 분류이며 CSV의 실제 재질 인증을 뜻하지 않습니다.

### 규격·제조사 자료에 따른 강종 코드 해석

| 강종 | 약어 풀이·영문 제품 분류 | 강종명 | 참고 규격·표기 |
| --- | --- | --- | --- |
| `SPCC` | Cold-Rolled Steel — Commercial Quality | 일반용 냉간압연 강판·강대 | JIS G 3141 [S1] |
| `SPCD` | Cold-Rolled Steel — Drawing Quality | 드로잉용 냉간압연 강판·강대 | JIS G 3141 [S1] |
| `SPCE` | Cold-Rolled Steel — Deep-Drawing Quality | 심가공용(딥드로잉용) 냉간압연 강판·강대 | JIS G 3141 [S1] |
| `SPCEN` | SPCE + Non-aging Guarantee | 비시효성 보증 심가공용 냉간압연 강판·강대 | SPCE에 N을 추가한 보증 표기 [S2]; 포스코 과거 자료의 SPCE(N) [P1] |
| `SGCC` | Hot-Dip Zinc-Coated Steel — Commercial Quality, Cold-Rolled Base | 냉연 원판 기반 일반용 용융아연도금 강판·강대 | JIS G 3302 [S4] |
| `SPHC` | Hot-Rolled Mild Steel — Commercial Quality | 일반용 열간압연 연강판·강대 | JIS G 3131 [S3] |
| `SPFH590` | Hot-Rolled High-Strength Steel with Improved Formability for Automobile Uses — 590 MPa Grade | 자동차용 가공성 개선 열간압연 고장력 강판·강대 | JIS G 3134; 인용 규격의 최소 인장강도 590 MPa [S3] |
| `DP590` | Dual Phase Steel — 590 MPa Tensile-Strength Class | 590 MPa급 복합조직강 | 일반적인 조직·강도 등급 표기; 포스코 자료는 590DP [P1] |

**표기 기준:** 영문은 문자별 공식 풀네임이 아니라 규격·제조사 자료의 제품 분류를 풀어 쓴 설명입니다. `DP`는 Dual Phase, `SPCEN`의 `N`은 비시효성 보증을 뜻합니다. SPCD는 Drawing, SPCE는 Deep-Drawing으로 구분하며 SPCE를 Extra Deep-Drawing으로 풀지 않습니다. DP590만으로 특정 JIS 규격·포스코 사내 제품과의 동등성을 확정하지 않습니다.

**포스코 자료와의 관계:** 과거 공개 카탈로그에는 SPCC·SPCD·SPCE(N)이 KS/JIS 대응 항목으로, 590DP가 제품 등급으로 제시됩니다. 이 표는 해당 자료와 기존 S1~S5를 참고한 코드 해석이며 최신 규격 원문을 대체하지 않습니다. CSV의 실제 적용 규격·판본과 제조사는 현재 자료로 확정할 수 없습니다. 분석용 원본 코드는 변경하지 않습니다.

[P1 포스코 자동차강판 카탈로그(2014 자료, DP강 및 부록 규격 대응표)](https://www.posco.com/homepage/docs/kor5/dn/product/info/k_car_pdf_2014.pdf)

**대표 용도:** SPCC는 가전·일반 성형, SPCD·SPCE·SPCEN은 자동차 패널 등 성형 부품, SGCC는 건축·가전 판재, SPHC는 일반 성형 부품, SPFH590·DP590은 자동차 구조·성형 부품에 사용됩니다(S1~S5).

**왜 이 비교가 중요한가?** 재질과 운전 조건에 따라 관측 수준이 달라질 수 있기 때문에 전체 자료를 한 기준으로 판단하기 전에 그룹 차이를 확인해야 합니다. 다만 현재 설정값·속도·두께가 없어 강종만의 영향은 분리할 수 없습니다. 문헌상의 소재 열처리 온도를 측정 대상이 미확정인 센서값과 직접 비교하지 않습니다.

| 질문 | 확인 방법 | 중요한 이유 |
| --- | --- | --- |
| 평균 온도가 다른가? | 같은 센서를 후보별·LOT별로 비교 | 서로 다른 센서의 온도 대역을 섞지 않기 위해 |
| 변동성이 다른가? | LOT 내부 표준편차와 후보별 합산 통계 비교 | LOT 간 수준 차이와 LOT 내부 흔들림을 구분하기 위해 |
| 차이가 반복되는가? | 같은 후보의 여러 LOT 비교 | 단일 LOT 현상을 강종 전체 특성으로 오인하지 않기 위해 |
| 낮은 평균은 지속 저하인가? | 실제 시간축에서 온도 확인 | 평균·최솟값만으로 지속시간을 알 수 없기 때문에 |

출처(기존 조사 자료): [S1 NIPPON STEEL 냉연](https://www.nipponsteel.com/product/catalog_download/pdf/U003en.pdf), [S2 Mycron SPCEN](https://www.mycronsteel.com/product-cold-rolled-coils.php), [S3 JFE 열연](https://www.jfe-steel.co.jp/en/products/sheets/catalog/b1e-001.pdf), [S4 JFE 도금](https://www.jfe-steel.co.jp/en/products/sheets/catalog/b1e-004.pdf), [S5 SSAB DP강](https://www.ssab.com/en-us/brands-and-products/ssab-docol/automotive-steel-grades/dual-phase-steel).

[↑ 목차](#toc)

### 열처리 배경과 이 데이터의 비교 범위

| 강종군 | 문헌에서 확인할 공정 배경 | 현재 온도로 판단할 수 있는 범위 |
| --- | --- | --- |
| SPCC·SPCD·SPCE·SPCEN | 냉연 후 소둔으로 조직·성형성을 조정. SPCEN의 N은 비시효성 보증 조건이지 고정 온도 레시피가 아님 [S1, S2] | 동일 센서의 관측 분포 비교. 등급 순서대로 소둔 온도가 높아진다고 해석하지 않음 |
| SGCC | 소둔과 용융아연도금은 구분되는 공정 [S4] | 이 온도 기록을 아연 도금조 온도와 직접 비교하지 않음 |
| SPHC·SPFH590 | 열연 제품 분류·기계적 성질에 관한 코드 [S3] | 열연 코드가 있다는 이유로 이 소둔로의 열처리 조건을 역산하지 않음 |
| DP590 | 복합조직 형성에는 가열·냉각 경로가 중요. 포스코 자료에 DP강 열사이클·조직 예시가 있음 [P1] | 590은 강도 등급. 측정 TC를 소재 열사이클·목표 온도로 간주하지 않음 |

8코드 모두 실제 목표 온도·유지시간·냉각 속도는 현재 파일에 없습니다. 수치 레시피를 임의로 채우지 않으며, **열처리 배경 조사는 완료하되 실제 조건 적합성은 판단 불가**로 기록합니다. 코드 정의의 출처는 위 S1~S5·P1, 센서 가정의 근거는 로컬 `data/EDA/00_데이터사전_T-CR1-CAL01_온도.md`입니다. 외부 자료 수집은 후속 과제가 아닙니다.


<a id="mapping"></a>
## 2. 데이터 준비와 LOT–강종 연결

핵심 분석은 **온도 파일과 조업이벤트 파일**을 사용합니다. 온도는 분석 대상이고 이벤트는 강종 후보를 붙이는 근거입니다. 5절에서는 후보 한 구간을 실제로 대조하기 위해 보유한 정기수리·CAL01 정비·교체 파일을 보조 사용합니다.

LOT 시작과 CR1 강종변경 시각이 같고 구간 내 변경이 1건일 때 AFT_VAL을 후보로 붙입니다. 적용 시점의 업무 의미가 미확정이므로 확정 라벨로 쓰지 않습니다. 기존 품질 검사와 연결 규칙을 유지합니다.


In [ ]:
# 확인하려는 것: 컴퓨터마다 경로가 달라도 같은 온도 파일을 찾도록 데이터 폴더를 정합니다.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 설치된 한글 글꼴을 사용해 그래프의 제목과 축 이름이 깨지지 않도록 합니다.
available_fonts = []
for font in font_manager.fontManager.ttflist:
    available_fonts.append(font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 한글 글꼴 설치 후 커널을 재시작하세요.")
plt.rcParams["axes.unicode_minus"] = False
from IPython.display import display

DATA_DIR = None
# 현재 폴더부터 상위 폴더까지 차례로 찾습니다.
search_folders = [Path.cwd()] + list(Path.cwd().parents)
for folder in search_folders:
    for candidate in [folder, folder / "data"]:
        if (candidate / "T-CR1-CAL01_온도.csv").is_file():
            DATA_DIR = candidate
            break
    if DATA_DIR is not None:
        break
if DATA_DIR is None:
    raise FileNotFoundError("프로젝트 data 폴더의 온도 CSV를 확인하세요.")

TEMP_CSV = DATA_DIR / "T-CR1-CAL01_온도.csv"
EVENT_CSV = DATA_DIR / "G-02_조업이벤트.csv"
REPAIR_CSV = DATA_DIR / "G-01_정기수리캘린더.csv"
pd.set_option("display.max_columns", 30)
print("데이터 폴더:", DATA_DIR)


In [ ]:
# 확인하려는 것: 온도와 이벤트에 어떤 항목이 들어 있는지 확인합니다. LOT 번호는 계산하지 않는 이름표로 읽습니다.
df_temp = pd.read_csv(TEMP_CSV, encoding="utf-8-sig", dtype={"LOT_NO": "string"})
df_event = pd.read_csv(EVENT_CSV, encoding="utf-8-sig", dtype="string")
print("온도 컬럼:", df_temp.columns.tolist())
print("이벤트 컬럼:", df_event.columns.tolist())

In [ ]:
# 확인하려는 것: 분석 전에 빠진 값과 중복 행이 얼마나 있는지 확인합니다.
quality_rows = []
for name, frame in [("온도", df_temp), ("조업이벤트", df_event)]:
    # 결측은 셀 개수, 중복은 모든 컬럼이 같은 행 개수입니다.
    missing_cells = frame.isna().sum().sum()
    duplicate_rows = frame.duplicated().sum()
    quality_rows.append({
        "자료": name, "행 수": len(frame), "열 수": len(frame.columns),
        "결측 셀": int(missing_cells),
        "완전 중복 행": int(duplicate_rows),
    })
quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)
display(df_temp.isna().sum().to_frame("온도 파일 컬럼별 결측"))
print("이벤트 ID 결측:", df_event["EVT_ID"].isna().sum())
print("이벤트 ID 중복:", df_event["EVT_ID"].dropna().duplicated().sum())

In [ ]:
# 확인하려는 것: 시간으로 데이터를 연결해도 되는지 확인합니다. 읽을 수 없는 시각과 중복 시각을 먼저 검사합니다.
time_rows = []
for name, frame, column in [("온도", df_temp, "MEAS_DT"), ("이벤트", df_event, "EVT_DT")]:
    original_time = frame[column].copy()
    parsed_time = pd.to_datetime(original_time, errors="coerce")
    time_rows.append({"자료": name, "원래 시각 결측": int(original_time.isna().sum()),
                      "변환 실패": int((original_time.notna() & parsed_time.isna()).sum())})
    frame[column] = parsed_time
time_quality = pd.DataFrame(time_rows)
display(time_quality)
# 같은 LOT에서 이전 행보다 시간이 뒤로 돌아간 경우를 셉니다.
time_difference = df_temp.groupby("LOT_NO")["MEAS_DT"].diff()
is_reverse = time_difference < pd.Timedelta(0)
raw_reverse_count = int(is_reverse.sum())
duplicate_time_count = int(df_temp.duplicated(["LOT_NO", "MEAS_DT"]).sum())
print("원본 LOT 내부 역순:", raw_reverse_count, "/ LOT·시각 중복:", duplicate_time_count)
# assert는 연결에 문제가 있으면 분석을 멈추는 검사입니다.
assert df_temp[["LOT_NO", "MEAS_DT"]].notna().all().all(), "온도 연결 키 결측 검토 필요"
assert df_event[["EVT_ID", "EVT_DT", "PLANT_CD", "EVT_TYPE"]].notna().all().all(), "이벤트 키 결측 검토 필요"
assert df_event["EVT_ID"].is_unique, "중복 이벤트 ID 검토 필요"
assert duplicate_time_count == 0, "중복 측정 시각 검토 필요"
df_temp = df_temp.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
df_event = df_event.sort_values("EVT_DT").reset_index(drop=True)

In [ ]:
# 확인하려는 것: 전체 공장과 CR1의 강종 수를 따로 셉니다. 변경 전후에 같은 코드가 있어도 한 번만 셉니다.
grade_events = df_event.loc[df_event["EVT_TYPE"] == "강종변경"].copy()
cr1_grade_events = grade_events.loc[grade_events["PLANT_CD"] == "CR1"].copy()
grade_scope_rows = []
for name, events in [("전체 공장", grade_events), ("CR1", cr1_grade_events)]:
    # 변경 전과 후 코드를 합친 다음 빈 값과 중복 코드를 제외합니다.
    combined = pd.concat([events["BEF_VAL"], events["AFT_VAL"]])
    combined = combined.dropna()
    codes = sorted(combined.unique())
    grade_scope_rows.append({"범위": name, "이벤트 수": len(events),
                             "전 종류": events["BEF_VAL"].nunique(),
                             "후 종류": events["AFT_VAL"].nunique(),
                             "전체 종류": len(codes), "강종 목록": ", ".join(codes)})
grade_scope = pd.DataFrame(grade_scope_rows)
display(grade_scope)
cr1_codes = pd.concat([cr1_grade_events["BEF_VAL"], cr1_grade_events["AFT_VAL"]])
cr1_codes = cr1_codes.dropna()
grade_list = sorted(cr1_codes.unique())

In [ ]:
# 확인하려는 것: 각 강종에서 바뀐 횟수와 각 강종으로 바뀐 횟수를 구분합니다. LOT 수를 세는 것은 아닙니다.
grade_counts = pd.DataFrame(index=pd.Index(grade_list, name="강종"))
grade_counts["변경 전 등장"] = cr1_grade_events["BEF_VAL"].value_counts()
grade_counts["변경 후 등장"] = cr1_grade_events["AFT_VAL"].value_counts()
grade_counts = grade_counts.fillna(0).astype(int)
display(grade_counts)

In [ ]:
# 확인하려는 것: LOT마다 관측 시작·끝·행 수·시간 간격·결측이 얼마나 다른지 확인합니다.
# 같은 LOT끼리 묶어 시작·끝·전체 행 수를 계산합니다.
lot_groups = df_temp.groupby("LOT_NO")
lot_summary = lot_groups.agg(
    start=("MEAS_DT", "min"), end=("MEAS_DT", "max"), rows=("MEAS_DT", "size")
).sort_values("start")
duration = lot_summary["end"] - lot_summary["start"]
lot_summary["duration_min"] = duration.dt.total_seconds() / 60
sampling = df_temp[["LOT_NO", "MEAS_DT"]].copy()
sampling["gap_sec"] = sampling.groupby("LOT_NO")["MEAS_DT"].diff().dt.total_seconds()
lot_gap_stats = sampling.groupby("LOT_NO")["gap_sec"].agg(["count", "median", "max"])
display(lot_summary.join(lot_gap_stats))
sensor_columns = df_temp.columns.drop(["LOT_NO", "MEAS_DT"])
# True(결측)를 더하면 LOT별 결측 셀 수가 됩니다.
missing_cells = df_temp[sensor_columns].isna()
missing_by_lot = missing_cells.groupby(df_temp["LOT_NO"]).sum()
display(missing_by_lot)

In [ ]:
# 확인하려는 것: LOT 시작과 강종변경 시각이 일치하는지 확인해 강종 후보를 붙입니다. 확정 라벨은 아닙니다.
lot_grade_review = lot_summary.copy()
for column in ["event_id", "before_grade", "candidate_grade"]:
    lot_grade_review[column] = pd.Series(pd.NA, index=lot_summary.index, dtype="string")
lot_grade_review["events_inside"] = 0
lot_grade_review["status"] = "미검토"
# LOT를 하나씩 꺼내 그 관측 기간 안에 있는 강종변경을 찾습니다.
for lot_id, row in lot_summary.iterrows():
    lot_start = row["start"]
    lot_end = row["end"]
    # between은 시작과 끝을 모두 포함합니다.
    is_inside = cr1_grade_events["EVT_DT"].between(lot_start, lot_end)
    events = cr1_grade_events.loc[is_inside]
    lot_grade_review.loc[lot_id, "events_inside"] = len(events)
    # 이벤트가 없거나 여러 개면 임의로 강종을 정하지 않습니다.
    if len(events) == 0:
        lot_grade_review.loc[lot_id, "status"] = "연결 불가: 구간 내 이벤트 없음"
    elif len(events) > 1:
        lot_grade_review.loc[lot_id, "status"] = "연결 보류: 구간 내 이벤트 여러 건"
    else:
        event = events.iloc[0]
        lot_grade_review.loc[lot_id, "event_id"] = event["EVT_ID"]
        lot_grade_review.loc[lot_id, "before_grade"] = event["BEF_VAL"]
        if event["EVT_DT"] != lot_start:
            lot_grade_review.loc[lot_id, "status"] = "연결 보류: 시작 시각 불일치"
        elif pd.isna(event["AFT_VAL"]):
            lot_grade_review.loc[lot_id, "status"] = "연결 불가: 변경 후 값 결측"
        else:
            lot_grade_review.loc[lot_id, "candidate_grade"] = event["AFT_VAL"]
            lot_grade_review.loc[lot_id, "status"] = "후보 연결: 시작 일치·적용 의미 미확인"
display(lot_grade_review)
unlinked_lots = lot_grade_review.loc[lot_grade_review["candidate_grade"].isna()]
display(unlinked_lots)


In [ ]:
# 확인하려는 것: 강종별로 실제 온도가 있는 LOT와 관측 행이 몇 개인지 확인합니다.
mapped = df_temp.copy()
# map은 LOT 번호를 기준으로 위에서 찾은 강종 후보를 가져옵니다.
mapped["candidate_grade"] = mapped["LOT_NO"].map(lot_grade_review["candidate_grade"])
assert len(mapped) == len(df_temp), "연결 과정에서 온도 행 수가 달라졌습니다."
coverage = mapped.groupby("candidate_grade").agg(lot_count=("LOT_NO", "nunique"), rows=("LOT_NO", "size"))
grade_coverage = grade_counts.join(coverage).fillna(0).astype(int)
display(grade_coverage)
print("미연결 LOT:", len(unlinked_lots), "/ 미연결 온도 행:", mapped["candidate_grade"].isna().sum())


**연결 결과를 읽는 법:** 14 LOT 모두 후보에 연결되지만 후보 종류는 7종입니다. 같은 후보의 LOT가 반복되고, 이벤트에 등장한 SPCC는 변경 후 온도 LOT가 없습니다. 이는 개수 오류나 SPCC 생산 부재를 뜻하지 않습니다. 14 LOT 연결표의 시작·끝·이벤트 ID가 후속 분석의 연결 근거입니다.

[↑ 목차](#toc)


#### 왜 강종 수와 LOT 수부터 구분했는가?

**왜 8종과 14 LOT가 다른 것이 중요한가?** 개수를 맞추려는 목적이 아니라 비교 단위를 잘못 정하지 않기 위해서입니다. 같은 강종의 반복 LOT는 ‘같은 후보에서 현상이 반복되는가’를 볼 기회입니다. 반대로 서로 다른 LOT를 서로 다른 강종으로 취급하면 실제로 없는 강종 차이를 만들게 됩니다.

**왜 시작 시각이 같아도 후보라고 부르는가?** 시간 일치는 연결의 근거지만 이벤트가 소재 투입·지시·센서 도달 중 무엇을 뜻하는지는 알려 주지 않습니다. 따라서 14 LOT의 연결 성공률이 100%여도 물리적 적용 시각이나 재질 인증 정확도가 100%라는 의미는 아닙니다.

**어떤 숫자는 중요도가 낮은가?** 이벤트 등장 횟수 자체는 온도 안정성이나 생산량을 평가하는 지표로는 직접적인 의미가 없습니다. SPCC의 후보 LOT 0도 온도가 0이거나 생산이 없었다는 뜻이 아닙니다. 이번 분석에서는 이벤트 빈도 순위보다 **실제로 관측된 LOT 수와 연결 근거**가 더 중요합니다.


<a id="statistics"></a>
## 3. 강종·LOT별 온도 통계

센서마다 유효수·평균·중앙값·최솟값·최댓값·표준편차를 계산합니다. 평균은 수준, 표준편차는 퍼짐이며 고장 점수가 아닙니다. 후보별 합산 통계는 긴 LOT에 더 영향을 받으므로 LOT별 통계와 동일 가중 평균도 함께 봅니다.


In [ ]:
# 확인하려는 것: 온도 센서만 골라 전체 센서 통계를 보존합니다.
temp_cols = ['CLN-IN']
for column in df_temp.columns:
    if column.startswith('TC-'):
        temp_cols.append(column)
stats = ['count', 'mean', 'median', 'min', 'max', 'std']
# groupby는 같은 후보·LOT끼리 묶고, agg는 목록의 통계를 각각 계산합니다.
lot_groups = mapped.groupby(['candidate_grade', 'LOT_NO'])[temp_cols]
lot_temp_stats = lot_groups.agg(stats)
grade_groups = mapped.groupby('candidate_grade')[temp_cols]
grade_temp_stats = grade_groups.agg(stats)
lot_means = mapped.groupby(['candidate_grade', 'LOT_NO'])[temp_cols].mean()
# 먼저 LOT 평균을 구한 후 평균을 내므로 긴 LOT에 더 큰 비중을 주지 않습니다.
equal_lot_means = lot_means.groupby('candidate_grade').mean()

# 대표 센서 TC-Z3로 읽는 법을 설명하고, 전체 센서는 아래 그림에서 확인합니다.
z3_summary = grade_temp_stats['TC-Z3'].copy()
z3_summary['lot_count'] = grade_coverage['lot_count']
z3_summary['equal_lot_mean'] = equal_lot_means['TC-Z3']
display(z3_summary.round(3))
display(lot_temp_stats['TC-Z3'].round(3))


**확인된 내용:** TC-Z3의 후보별 합산 평균·표준편차에는 차이가 있습니다. DP590은 평균 약 855.850, 중앙값 약 860.205로 평균이 약 4.36 낮습니다. 낮은 값의 영향을 검토할 근거지만 얼마나 오래 낮았는지는 시계열로 확인해야 합니다. SGCC·DP590·SPCD는 각 1 LOT여서 반복성을 판단할 수 없습니다.

`grade_temp_stats`는 후보별 전체 행 통계, `lot_temp_stats`는 LOT별 통계, `equal_lot_means`는 LOT 하나당 같은 비중의 평균입니다. NaN·빈 관측을 0이나 정상으로 바꾸지 않습니다.

[↑ 목차](#toc)


#### 왜 평균 하나만으로 결론을 내리지 않는가?

| 질문 | 실제 결과에서 얻은 답 | 왜 중요한가·어디까지 해석하는가 |
| --- | --- | --- |
| 후보 평균의 최대·최소 차이는 큰가? | TC-Z3 행 가중 평균 차이는 약 4.87 | 관측 차이가 있다는 사실은 명확하지만 허용오차·설정값이 없어 공정상 큰 차이 또는 이상이라고 평가할 수 없음 |
| 왜 중앙값도 보는가? | DP590 평균 855.850, 중앙값 860.205 | 평균이 낮은 쪽 관측의 영향을 받았음을 살펴볼 단서. 낮은 값이 몇 초인지 몇 시간인지는 이 두 숫자로 알 수 없음 |
| 왜 LOT 동일 가중 평균도 보는가? | SPHC 전체 행 평균 858.541, LOT 동일 가중 평균 857.832 | 전체 행 평균은 관측량 많은 LOT의 영향이 큼. 두 값 중 하나가 정답인 것이 아니라 각각 ‘관측 행’과 ‘LOT’를 같은 비중으로 본 다른 질문 |
| 왜 최솟값만으로 고장을 고르지 않는가? | SPHC 240061 최솟값 812.657, 전체 평균 852.455 | 한 점의 크기와 지속적인 저하는 다름. 센서 유효성·지속시간·인접 센서를 확인해야 하며 최솟값 순위는 원인 순위가 아님 |
| 왜 유효 관측수를 보는가? | LOT별 길이와 센서별 결측이 다름 | 동일한 시간과 독립 반복 횟수를 관측한 것이 아님. 초 단위 행 수가 많아도 같은 LOT 하나의 일반화 한계는 남음 |

**따라서 활용할 결론:** 후보 평균으로 검토 대상을 좁힌 뒤 LOT별 통계와 시계열로 내려갑니다. 평균 차이를 유의성 검정이나 고장 판정으로 바꾸지는 않았습니다. 실제 운전 조건이 없는 만큼 이번 결과의 가치는 원인 확정보다 **어느 LOT의 어떤 구간을 설명해야 하는지 찾는 데** 있습니다.


<a id="plots"></a>
## 4. 시각화로 확인하는 차이

### 그림 1. 같은 후보에서도 LOT의 수준·변동성이 같은가?

TC-Z3는 해석 예시 센서이며 전체 공정을 대표한다고 가정하지 않습니다. 점 하나는 LOT 하나, 숫자는 LOT 번호입니다. 왼쪽은 평균, 오른쪽은 LOT 내부 표준편차입니다.


In [ ]:
# 확인하려는 것: 강종 간 차이와 같은 강종 후보 안의 LOT 차이를 나눠 봅니다.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# 왼쪽은 평균, 오른쪽은 표준편차입니다.
metrics = ['mean', 'std']
titles = ['평균 온도', '온도 표준편차']
observed_grades = lot_temp_stats.index.get_level_values('candidate_grade')
for panel in range(2):
    ax = axes[panel]
    metric = metrics[panel]
    title = titles[panel]
    for position, grade in enumerate(grade_list):
        if grade not in observed_grades:
            continue
        values = lot_temp_stats.loc[grade, ('TC-Z3', metric)]
        # 각 LOT의 값을 점으로 찍고 옆에 LOT 번호를 표시합니다.
        for lot, value in values.items():
            ax.scatter(position, value)
            ax.annotate(str(lot), (position, value), xytext=(3, 3), textcoords='offset points', fontsize=7)
    ax.set_xticks(range(len(grade_list)), grade_list, rotation=30)
    ax.set_title('강종 후보·LOT별 TC-Z3 ' + title)
    ax.set_ylabel(title + ' (기록값)')
    ax.set_xlabel('강종 후보')
    ax.grid(axis='y', alpha=.2)
fig.tight_layout()
plt.show()


**실제 결과:** SPHC 후보 4 LOT의 TC-Z3 표준편차는 약 1.827~11.303으로 다릅니다. 같은 후보 안에서도 차이가 커서 후보별 합산 표준편차 5.185 하나로 모든 SPHC LOT를 설명할 수 없습니다. SPCEN의 두 LOT도 약 3.574와 9.864로 다릅니다. 따라서 강종별 통계 다음에 LOT별 검토가 필요합니다.

**확실히 볼 수 있는 것:** 관측 LOT의 평균 수준과 산포, 같은 후보에 속한 LOT들 사이의 차이입니다. SPCC에는 관측점이 없습니다. 점이 하나뿐인 후보에서는 재현성을 볼 수 없습니다.

**다음 확인:** 산포가 큰 점의 LOT를 시계열로 확인합니다. 같은 후보에서도 평균이 다르면 강종 외 차이가 섞여 있음을 고려합니다. 후보 간 점의 차이는 인과 효과나 통계적 유의성의 증거가 아닙니다.


#### 그림 1을 보며 묻는 질문

**왜 같은 강종 안의 점들을 먼저 보는가?** SPHC의 LOT 평균은 852.455~860.059로 약 7.60 차이입니다. 이는 후보별 합산 평균 범위 약 4.87보다도 큽니다. 서로 다른 집계 단위의 기술적 비교이지만, 적어도 강종 이름만으로 각 LOT의 온도 수준을 충분히 설명할 수 없다는 점을 보여 줍니다. 어떤 원인이 더 크다는 분산분석 결과는 아닙니다.

**왜 평균과 표준편차를 나란히 그렸는가?** 낮은 평균이 모든 시각에 고르게 낮았기 때문인지, 시간에 따른 수준 이동이 섞였기 때문인지 구분할 실마리를 얻기 위해서입니다. SPHC 240061은 평균이 낮고 표준편차가 가장 커 우선 시계열 검토 대상으로 선택됩니다. ‘가장 위험한 LOT’가 아니라 **현재 선택 기준에서 가장 설명이 필요한 LOT**입니다.

**반복 관측에서 무엇이 보이는가?** SPCE 3 LOT 평균은 860.311~860.809로 약 0.50 차이지만 표준편차는 1.974~3.684입니다. 평균이 비슷해도 내부 흔들림까지 같지는 않습니다. 관측된 세 LOT의 결과이며 SPCE가 항상 안정적이라는 의미는 아닙니다.

**왜 점 하나의 강종 순위는 덜 중요하게 보는가?** DP590·SGCC·SPCD는 한 점씩이어서 ‘강종 차이’와 ‘그 LOT만의 차이’가 겹칩니다. 그래프에서 낮거나 높다는 순위보다 관측 반복 수를 먼저 봅니다. LOT 번호는 해당 시계열을 찾아가는 이름표이지 온도에 영향을 주는 수치 변수가 아닙니다.

### 그림 2. 차이는 어느 온도 센서에 나타나는가?

각 패널은 센서 하나이며 점은 LOT 평균입니다. 세로축 범위는 센서마다 다르므로 패널 간 점의 높이로 차이의 크기를 비교하지 않습니다.


In [ ]:
# 확인하려는 것: TC-Z3에서 본 차이가 다른 센서에서도 나타나는지 확인합니다.
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
# 4행×3열의 그래프를 한 줄 목록으로 바꿔 센서 하나씩 배치합니다.
plot_axes = axes.flatten()
for sensor_number, sensor in enumerate(temp_cols):
    ax = plot_axes[sensor_number]
    for position, grade in enumerate(grade_list):
        if grade in lot_means.index.get_level_values('candidate_grade'):
            values = lot_means.loc[grade, sensor]
            ax.scatter([position] * len(values), values, s=25)
    ax.set_xticks(range(len(grade_list)), grade_list, rotation=45, fontsize=8)
    ax.set_title(sensor + '의 LOT 평균 온도')
    ax.set_ylabel('온도 기록값')
    ax.grid(axis='y', alpha=.2)
fig.tight_layout()
plt.show()


**확실히 볼 수 있는 것:** 어떤 센서에서 관측 평균들이 겹치거나 떨어지는지, 같은 후보 내부에도 LOT 간 차이가 있는지입니다. 각 점은 평균이므로 여러 센서가 동시에 변했다는 사실까지 알려 주지는 않습니다.

**다음 확인:** 차이가 큰 센서를 `lot_temp_stats`의 표준편차·범위와 함께 보고 실제 시각을 맞춰 비교합니다. 센서별 단위·측정 대상의 가정을 유지하며 문헌의 권장 온도와 직접 대조하지 않습니다.


#### 그림 2를 보며 묻는 질문

**왜 센서를 나눠 보는가?** 서로 다른 온도 대역을 한 평균으로 합치면 물리적 의미가 흐려집니다. LOT 동일 가중 후보 평균 기준 TC-Z1은 약 99.606~100.000, TC-Z3은 855.850~860.699, TC-OUT1은 276.756~285.568입니다. 같은 후보 구분이라도 센서에 따라 관측 차이의 크기와 순서가 다릅니다. 이 수치는 그래프의 LOT 점들을 후보별로 동일 가중한 요약입니다.

**왜 축의 눈금이 중요한가?** 패널별 세로축이 자동으로 달라 작은 차이도 화면에서는 크게 보일 수 있습니다. TC-Z1의 점들이 떨어져 보여도 후보 평균 범위는 약 0.39입니다. 화면상 거리만으로 TC-Z3보다 차이가 크다고 읽으면 안 됩니다. 허용 기준이 없으므로 0.39가 공정상 중요하지 않다고 단정할 수도 없습니다.

**왜 12개 센서 그림만으로 동시 변화를 말할 수 없는가?** 각 점은 LOT 전체 평균으로 시간 정보가 압축되어 있습니다. 두 센서 평균이 함께 낮아도 같은 시각에 하락했는지는 모릅니다. 후속 작업은 240061의 TC-Z3~Z6를 같은 MEAS_DT에 맞춰 비교하는 것입니다.

**어디에 활용하는가?** 상세 시계열을 볼 센서를 고르는 데 사용합니다. 평균 순위로 강종의 열처리 온도를 추천하거나, TC-Z1의 차이가 작다는 이유로 해당 센서를 제거하는 근거로 사용하지 않습니다.

### 그림 3. 큰 변동성은 시간축에서 어떻게 나타나는가?

TC-Z3의 LOT 내부 표준편차가 가장 큰 LOT를 계산으로 선택합니다. 이는 시계열 검토 사례를 고르는 규칙이며 고장 판정이 아닙니다. 이벤트·정비에 의한 원인은 다음 담당자가 검토합니다.


In [ ]:
# 확인하려는 것: 변동성이 가장 큰 LOT에서 온도 저하의 시점과 출력 동반 변화를 봅니다.
# idxmax는 표준편차가 가장 큰 행의 (강종 후보, LOT 번호)를 돌려줍니다.
z3_lot_std = lot_temp_stats[('TC-Z3', 'std')]
selected_grade, selected_lot = z3_lot_std.idxmax()
is_selected_lot = mapped['LOT_NO'] == selected_lot
part = mapped.loc[is_selected_lot].copy()
part = part.sort_values('MEAS_DT')
gaps = part['MEAS_DT'].diff().dt.total_seconds()
# 첫 행이거나 시간 간격이 평소의 3배보다 길면 새 구간으로 나눕니다.
# cumsum은 새 구간이 시작될 때마다 구간 번호를 1씩 올립니다.
is_first_row = gaps.isna()
is_long_gap = gaps > gaps.median() * 3
start_new_segment = is_first_row | is_long_gap
part['segment'] = start_new_segment.cumsum()
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for _, segment in part.groupby('segment'):
    axes[0].plot(segment['MEAS_DT'], segment['TC-Z3'], linewidth=.7)
    axes[1].plot(segment['MEAS_DT'], segment['OP-Z3'], linewidth=.7)
axes[0].set_title(f'{selected_grade} 후보 LOT {selected_lot}: 온도 변화')
axes[1].set_title('같은 시각의 제어출력 변화 — 원인 판단을 위한 보조 관측')
axes[0].set_ylabel('TC-Z3 기록값')
axes[1].set_ylabel('OP-Z3 기록값')
axes[1].set_xlabel('측정 시각')
fig.tight_layout()
plt.show()
display(lot_temp_stats.loc[(selected_grade, selected_lot), 'TC-Z3'].to_frame('선택 LOT 통계'))


**실제 결과:** 최대 표준편차 사례는 SPHC 후보 LOT 240061입니다. TC-Z3 평균 약 852.455, 중앙값 857.143, 표준편차 11.303입니다. 그림에서 관측 초반 약 813 수준에서 점차 상승하는 추세와 이후 흔들림이 보입니다. 따라서 큰 표준편차에는 시간에 따른 수준 변화가 포함되며, 일정한 온도 주변의 무작위 흔들림만을 의미하지 않습니다. OP-Z3에는 0 근처와 높은 출력 구간이 반복되지만 이것만으로 제어 원인이나 고장을 단정할 수 없습니다.

**확실히 볼 수 있는 것:** 선택한 LOT 안의 실제 온도 변화 시점·모양과 같은 시각의 출력입니다. 긴 수집 간격은 선을 끊었으며 선이 끊겼다고 설비가 정지했다는 뜻은 아닙니다.

**다음 확인:** 낮은 구간의 시작·끝·지속시간을 기록하고 같은 구간의 다른 TC·OP와 결측을 확인합니다. 출력 반응의 방향·지연·원인과 고장 여부는 이 그림만으로 확정할 수 없습니다.

[↑ 목차](#toc)


#### 그림 3을 보며 묻는 질문

**왜 이 LOT의 표준편차가 컸는가?** 그림에서 초반 낮은 수준에서 점차 상승하는 추세가 보입니다. 전체 표준편차 11.303은 첫 10분의 4.903, 마지막 10분의 3.055보다 큽니다. 전체 값의 퍼짐에 구간 간 수준 차이가 포함되기 때문입니다. 다만 두 10분 구간만으로 전체 산포의 원인을 모두 설명하거나 각 원인의 기여율을 계산한 것은 아닙니다.

**왜 첫·마지막 평균 차이 약 37.31이 중요한가?** 이 LOT를 하나의 일정한 수준으로 간주하기 어렵다는 구체적 증거입니다. 같은 LOT 안에서도 시간 구간을 구분해 비교해야 함을 보여 줍니다. 후반을 정상으로 확정한 것이 아니므로 ‘37.31만큼 회복했다’보다 ‘후반 평균이 37.31 높았다’라고 표현합니다.

**왜 출력 그래프를 함께 봤는가?** 온도 변화와 제어 동작이 함께 나타나는지 탐색하기 위해서입니다. 하지만 첫 10분 OP-Z3 결측은 179/589행(약 30.4%)입니다. 온도 평균은 589행, 출력 평균은 410행에 근거하므로 두 평균의 관계만으로 제어 효율을 평가하면 관측 시각 차이를 놓칩니다. 후속 비교는 두 센서가 동시에 유효한 행으로 제한해야 합니다.

**왜 출력 0이나 높은 값을 바로 이상이라고 하지 않는가?** 0은 이 데이터에서 결측이 아닌 실제 기록값입니다. 제어 대상·설정값이 없으므로 값의 크기만으로 정지나 포화를 확정할 수 없습니다. 현재 중요한 것은 해당 값의 지속시간과 온도·다른 센서와의 시간 관계입니다.

**왜 이 그래프가 강종변경 전후 효과의 증거는 아닌가?** 관측은 변경 시각부터 시작합니다. 초반보다 앞선 온도가 없으므로 이전 DP590과 이후 SPHC를 비교한 그림이 아닙니다. 이 그림은 SPHC 후보 한 LOT 내부 변화만 보여 줍니다.


<a id="handoff"></a>
## 5. 결론과 다음 분석자 인계

**이번 분석으로 알 수 있는 것:** 강종 후보별 관측 온도는 같지 않으며, LOT별 평균과 내부 변동성을 구분해 볼 필요가 있습니다. 전체 행 평균은 관측량의 영향을 받습니다. 강종이 같아도 같은 운전 조건이라는 보장이 없고, 단일 LOT 후보에서는 강종과 LOT 영향을 분리할 수 없습니다.

**이번에 결론 내리지 않는 것:** 확정 강종 라벨, 강종 단독 인과 효과, 정상 레시피, 고장 원인·성능. 외부 데이터 수집을 요구하지 않고 현재 자료의 한계로 남깁니다.

### 변경 전후 관측 가능성 재확인

다음 담당자가 관측되지 않은 전후 효과를 계산하지 않도록 CR1 이벤트 전체의 ±10분 관측량을 남깁니다. 빈 관측을 보간하거나 다른 날짜 LOT로 대체하지 않습니다.


In [ ]:
# 확인하려는 것: 변경 전후 분석을 현재 자료로 수행할 수 있는지 확인합니다.
WINDOW = pd.Timedelta(minutes=10)
times = df_temp['MEAS_DT'].sort_values()
rows = []
for event in cr1_grade_events.itertuples(index=False):
    # 변경 전에는 변경 시각을 제외하고, 변경 후에는 포함합니다.
    before_start = event.EVT_DT - WINDOW
    after_end = event.EVT_DT + WINDOW
    is_before = (times >= before_start) & (times < event.EVT_DT)
    is_after = (times >= event.EVT_DT) & (times <= after_end)
    before_count = is_before.sum()
    after_count = is_after.sum()
    rows.append({'EVT_ID': event.EVT_ID, 'before_rows': before_count, 'after_rows': after_count})
event_availability = pd.DataFrame(rows)
print('전체 변경:', len(event_availability))
print('전 관측 있는 변경:', event_availability['before_rows'].gt(0).sum())
print('후 관측 있는 변경:', event_availability['after_rows'].gt(0).sum())
print('양쪽 관측:', (event_availability['before_rows'].gt(0) & event_availability['after_rows'].gt(0)).sum())


### 기존 다른 자료는 다음에 어떻게 분석하는가?

핵심 자료의 큰 틀은 온도·조업이벤트·정기수리캘린더입니다. 이번에는 강종 연결에 필요한 이벤트만 직접 사용하고, 아래 사건 대조 방법 중 후보 한 구간은 하단에서 실행하며, 다른 구간으로의 확장은 다음 담당자에게 인계합니다. 기존 상세본의 정비 계산 결과는 인계 참고이며 아래 원인 분석의 완료를 뜻하지 않습니다.

| 보유 자료·사용 이유 | 연결 방법 | 현재 가능한 다음 작업 | 남길 산출물·한계 |
| --- | --- | --- | --- |
| CR1 조업이벤트: 소재 변경과 동시 발생했는지 확인 | PLANT_CD=CR1, EVT_DT, BEF_VAL·AFT_VAL | 후보 구간 앞뒤의 변경 시각·값 표시 | 이벤트 대조표. ±10분 전후 효과는 전 관측 부재로 제외 |
| 정기수리캘린더: 계획 일정과 공백 구분 | CR1 STRT_DT~END_DT; 날짜 종료일 전체 포함 가정 | 후보·관측 구간과 계획수리 중첩 확인 | 중첩 표. 계획을 실제 정지로 단정하지 않음 |
| 설비마스터: 분석 대상 혼동 방지 | EQP_TAG·PARENT_TAG·PLANT_CD | CAL01 소속과 대상 기록 확인 | 대상 필터 근거. 센서 위치까지 확정하지 않음 |
| 정비이력: 계획 외 개별 작업 시각 대조 | EQP_TAG=P1-CR1-CAL01, DETC_DT·STRT_DT·END_DT | 후보 전후에 발견·작업이 있었는지 조회 | 사건 시각·차이·정비 유형. 정비가 곧 고장은 아님 |
| 부품교체이력: 작업의 교체 대상 확인 | WO_NO로 정비와 연결 | 해당 작업의 부품·교체 사유 검토 | 작업별 상세 표. 센서 원본에 다중 조인하여 행을 늘리지 않음 |

**다음 담당자의 실행 순서**

1. 위에서부터 실행해 `lot_grade_review`의 연결 근거와 `grade_coverage`의 관측 범위를 확인합니다.
2. `lot_temp_stats`와 그림 1에서 검토할 센서·LOT를 선택합니다. 그림 3은 최대 표준편차 사례를 기본 제공합니다.
3. 후보 구간마다 **LOT·강종 후보·센서·시작·끝·선정 이유·유효 관측 수**를 남깁니다. 후보 기준과 계산 창을 명시하며 고장 라벨로 쓰지 않습니다.
4. 같은 후보의 기존 LOT, 같은 시각의 다른 존·OP·결측을 대조합니다. 이동통계와 실제 시각 기반 지연 분석은 후속 작업입니다.
5. 위 표의 기존 이벤트·정기수리·정비·교체와 대조해 **관련 기록 ID·시각 차이·일치/미일치·판단 한계**를 기록합니다. 다른 공장은 제외합니다.

**모델링 인계:** 먼저 설명 가능한 후보 기준을 앞선 LOT에서 정하고 뒤의 LOT에 적용합니다. 행 무작위 분할·전체 자료로 정한 임곗값·미래를 포함한 이동통계·사후 정비 사유의 입력 사용을 피합니다. 현재는 후보 경보 수·지속시간·관측시간 대비 빈도까지 검토하며 검증된 고장 라벨이 없으므로 정확도·재현율·잔여수명을 주장하지 않습니다. 이 노트북은 모델 학습을 실행하지 않습니다.

[↑ 목차](#toc)


### 실제 검토 구간과 주변 사건 대조

그림 3에서 선택한 **TC-Z3 표준편차 최대 LOT의 첫 10분**을 검토 구간으로 정의합니다. 초반 상승 추세가 전체 표준편차에 기여하는지 확인하려는 기술적 비교이며 고장 임곗값이 아닙니다. 같은 LOT 마지막 10분과 비교하되 강종변경 전후 효과나 정상화 효과라고 부르지 않습니다. 마지막 구간도 정상으로 인증된 자료가 아닙니다.

이벤트는 후보 구간의 ±10분과 넓은 문맥용 ±24시간을 따로 조회합니다. 정비는 CAL01의 발견 시각 또는 작업 기간 중첩, 정기수리는 CR1 계획 기간 중첩을 봅니다. 부품교체는 관련 정비의 WO_NO로만 연결합니다. 이 보조 자료는 온도 변화 근처의 사건을 확인하기 위해 사용하며 온도 원본에 직접 합치지 않습니다.


In [ ]:
# 확인하려는 것: 높은 표준편차가 있는 LOT의 초반과 후반 수준을 실제 값으로 비교합니다.
candidate_start = part['MEAS_DT'].min()
candidate_end = candidate_start + pd.Timedelta(minutes=10)
last_end = part['MEAS_DT'].max()
# 기존 계산과 똑같이 첫 구간은 [시작, 끝), 마지막 구간은 (시작, 끝]을 씁니다.
is_first_window = (part['MEAS_DT'] >= candidate_start) & (part['MEAS_DT'] < candidate_end)
last_start = last_end - pd.Timedelta(minutes=10)
is_last_window = (part['MEAS_DT'] > last_start) & (part['MEAS_DT'] <= last_end)
first_values = part.loc[is_first_window]
last_values = part.loc[is_last_window]
candidate_record = pd.DataFrame([{'LOT_NO': selected_lot, '강종 후보': selected_grade,
    '시작(포함)': candidate_start, '끝(미포함)': candidate_end,
    '선정 이유': 'TC-Z3 표준편차 최대 LOT의 초반 상승 구간 검토; 고장 판정 아님'}])
display(candidate_record)
window_rows = []
for label, window in [('첫 10분', first_values), ('마지막 10분', last_values)]:
    for sensor in ['TC-Z3', 'OP-Z3']:
        values = window[sensor]
        window_rows.append({'구간': label, '센서': sensor, '전체 행': len(window),
            '유효수': values.count(), '결측수': values.isna().sum(), '평균': values.mean(),
            '중앙값': values.median(), '최솟값': values.min(), '최댓값': values.max(), '표준편차': values.std()})
candidate_window_stats = pd.DataFrame(window_rows)
display(candidate_window_stats.round(3))



In [ ]:
# 확인하려는 것: 후보와 대조할 CAL01 정비·교체·CR1 계획수리만 준비합니다.
# 후보 근처에 기록된 조업·정비·계획수리가 실제로 있는지 확인합니다.
maintenance = pd.read_csv(DATA_DIR / 'G-02_정비이력.csv', dtype='string')
jobs = maintenance.loc[maintenance['EQP_TAG'] == 'P1-CR1-CAL01'].copy()
for column in ['DETC_DT', 'STRT_DT', 'END_DT']:
    jobs[column] = pd.to_datetime(jobs[column], errors='raise')
replacement = pd.read_csv(DATA_DIR / 'G-02_부품교체이력.csv', dtype='string')
plans = pd.read_csv(REPAIR_CSV, dtype='string')
plans = plans.loc[plans['PLANT_CD'] == 'CR1'].copy()
for column in ['STRT_DT', 'END_DT']:
    plans[column] = pd.to_datetime(plans[column], errors='raise')


In [ ]:
# 확인하려는 것: 조회 범위별 사건 시각·변경 값과 기록 건수를 확인합니다.
context_counts = []
candidate_context = {}
for label, margin in [('근접 ±10분', pd.Timedelta(minutes=10)), ('문맥 ±24시간', pd.Timedelta(hours=24))]:
    # 후보 구간 앞뒤로 여유 시간을 더해 주변 사건을 찾습니다.
    lo = candidate_start - margin
    hi = candidate_end + margin
    is_cr1 = df_event['PLANT_CD'] == 'CR1'
    is_near_time = df_event['EVT_DT'].between(lo, hi)
    events = df_event.loc[is_cr1 & is_near_time].copy()
    events['후보 시작과 차이(분)'] = (events['EVT_DT'] - candidate_start).dt.total_seconds() / 60
    # 발견 시각이 가깝거나 작업 기간이 겹치면 관련 기록으로 조회합니다.
    found_nearby = jobs['DETC_DT'].between(lo, hi)
    work_overlaps = (jobs['STRT_DT'] <= hi) & (jobs['END_DT'] >= lo)
    near_jobs = jobs.loc[found_nearby | work_overlaps]
    # isin은 조회된 작업번호 목록에 들어 있는 교체 기록인지 확인합니다.
    related_work_numbers = near_jobs['WO_NO']
    is_related_part = replacement['WO_NO'].isin(related_work_numbers)
    near_parts = replacement.loc[is_related_part]
    # 종료일이 날짜만 있으므로 다음 날 0시 직전까지 계획 기간으로 봅니다.
    plan_end_exclusive = plans['END_DT'] + pd.Timedelta(days=1)
    plan_overlaps = (plans['STRT_DT'] <= hi) & (plan_end_exclusive > lo)
    near_plans = plans.loc[plan_overlaps]
    candidate_context[label] = {'조업이벤트': events, '정비': near_jobs, '교체': near_parts, '정기수리': near_plans}
    context_counts.append({'조회 범위': label, '조업이벤트': len(events), '정비': len(near_jobs),
                           '교체 상세': len(near_parts), '정기수리': len(near_plans)})
    print(label)
    display(events[['EVT_ID', 'EVT_DT', 'EVT_TYPE', 'BEF_VAL', 'AFT_VAL', '후보 시작과 차이(분)']])
    display(near_jobs[['WO_NO', 'DETC_DT', 'STRT_DT', 'END_DT', 'MNT_TYPE', 'FAIL_MODE']])
    display(near_parts)
    display(near_plans)
context_counts = pd.DataFrame(context_counts)
display(context_counts)


### 실제 대조 결과 해석

검토 구간은 **SPHC 후보 LOT 240061, 2024-06-15 01:04:00 이상~01:14:00 미만**입니다. 온도 통계 비교는 같은 LOT 내 첫·마지막 10분이며 강종변경 전후 비교가 아닙니다.

| 확인한 값·사건 | 결과 | 해석 |
| --- | --- | --- |
| TC-Z3 평균 | 첫 10분 820.198 → 마지막 10분 857.505 | 약 37.31의 수준 차이. 초반 상승이 전체 표준편차에 기여하며 고장 여부는 미확정 |
| TC-Z3 표준편차 | 첫 10분 4.903, 마지막 10분 3.055; LOT 전체 11.303 | 전체 산포는 각 짧은 구간의 흔들림보다 크며 시간에 따른 수준 차이를 포함 |
| OP-Z3 평균 | 첫 10분 73.276, 마지막 10분 68.832 | 서로 다른 온도 수준의 구간이고 초반 결측이 많아 약 4.44 감소를 효율 개선으로 해석할 수 없음 |
| OP-Z3 유효수·결측 | 첫 10분 유효 410·결측 179, 마지막 유효 586·결측 0 | 결측을 0으로 채우지 않음. 후속 상관·지연 분석은 유효 쌍 수와 결측 구간을 명시해야 함 |
| 강종변경 EVT-2024-0122 | 2024-06-15 01:04, DP590→SPHC | 후보 시작과 정확히 일치. 이전 온도가 없으므로 강종변경 때문에 상승했다고 판단 불가 |
| CAL01 정비 WO-2024-00972 | 발견 6/14 14:09, 작업 15:35~18:03, MNT_TYPE=TBM | 후보 시작은 작업 종료 약 7시간 1분 후. 실제 재가동 시각은 없으므로 기동 구간으로 확정 불가 |
| CR1 정기수리 PLAN-2024-02 | 6/10~6/14, 연속소둔로 노내 정비 | 후보는 계획 종료일 다음 날. 날짜 종료일 전체 포함 가정 아래 ±24시간 문맥에 걸림; 후보 자체의 계획수리 중첩을 뜻하지 않음 |
| 주변 교체 상세 | 선택한 정비의 WO_NO에 대응하는 교체 0행 | 해당 조회에 교체 기록 없음. 설비 전체에서 교체가 없었다고 일반화하지 않음 |

±10분 조회에서는 강종변경 1건만, ±24시간 조회에서는 강종변경 1건·정비 1건·정기수리 1건이 확인됩니다. **주변 사건이 있었다는 사실과 온도 변화의 원인을 입증하는 것은 다릅니다.** TBM 코드의 정확한 정의·미기록 작업 여부·실제 가동 상태는 현재 자료로 확정하지 않습니다.

### 왜 주변 사건은 중요하지만 원인으로 확정하지 않는가?

**왜 정기수리와 개별 정비를 함께 보는가?** 계획수리는 일정, 정비이력은 개별 작업의 기록으로 의미가 다릅니다. 계획 종료 다음 날의 온도 관측이고 개별 작업 종료 약 7시간 1분 뒤라는 맥락은 초반 추세를 조사할 이유가 됩니다. 그러나 그 사이 실제로 언제 가동했는지 모릅니다. 따라서 ‘정비 후 가까운 관측’은 확인됐지만 ‘재가동으로 인한 승온’은 아직 가설입니다.

**왜 ±10분과 ±24시간을 구분하는가?** ±10분은 후보 근처 기록, ±24시간은 더 넓은 배경을 찾는 탐색 창입니다. 넓히면 사건이 더 걸릴 가능성도 커집니다. 24시간 내 정비가 있다는 이유만으로 관련성이 더 강하다고 판단할 수 없습니다. 다른 기존 LOT에도 같은 창을 적용해 비슷한 추세와 사건이 반복되는지 확인해야 합니다.

**왜 교체 기록 0행을 중요하게 과장하지 않는가?** 해당 조회에서 교체 원인을 구체화할 근거가 없다는 뜻입니다. 설비에 문제가 없었다는 증거나 모든 교체가 없었다는 증거는 아닙니다. 반대로 기록이 없는 상황에서 부품 고장 이야기를 추가할 근거도 없습니다.

**왜 다음 분석을 결측 확인부터 시작하는가?** 결측 상태를 무시한 온도–출력 비교는 서로 다른 시각의 평균을 연결할 수 있기 때문입니다. 현재 자료로 검토할 우선순위는 **유효 관측 확인 → 같은 시각 TC·OP 비교 → 반복 LOT 비교 → 주변 사건의 반복 여부 확인**입니다.

### 결과별로 구체적으로 어떤 데이터와 함께 보는가?

핵심 자료는 **온도·조업이벤트·정기수리캘린더**입니다. 정비·교체·마스터는 아래 질문에 필요한 경우에만 보조 사용합니다. 모두 현재 `data/`에 있는 파일이며, 타 공정 데이터나 외부 자료를 추가로 확보하는 계획은 아닙니다.

| 출발할 분석 결과·질문 | 함께 볼 파일·컬럼 | 연결 기준·보는 순서 | 확인할 내용과 남길 결과 |
| --- | --- | --- | --- |
| LOT 240061의 초반 TC-Z3가 낮고 점차 상승함. 다른 존도 함께 변했는가? | `T-CR1-CAL01_온도.csv`: LOT_NO, MEAS_DT, TC-Z3~Z6, TC-Z7~Z9, TC-OUT1~OUT2 | LOT_NO=240061로 제한하고 **같은 MEAS_DT**의 센서를 각각 그림. 먼저 01:04~01:14, 이후 LOT 전체 확인 | 여러 존의 동반 변화인지 TC-Z3 중심 현상인지. 센서별 시작·끝·평균·변동을 기록하며 서로 다른 온도 대역을 합쳐 평균 내지 않음 |
| 온도 상승 때 제어출력은 어떻게 움직였는가? | 같은 온도 파일: TC-Z3·OP-Z3, 이후 TC-Z4·OP-Z4 등 같은 번호의 쌍 | 같은 LOT·시각의 **두 값이 모두 있는 행**만 비교. TC-Z9에는 대응 OP-Z9가 없음 | 동시 변화·출력 0 또는 높은 값의 지속·시간 지연 후보. 제어 방향과 원인을 확정하지 않음 |
| 초반 OP-Z3 결측 179행이 출력 해석을 바꾸는가? | 같은 파일: MEAS_DT, OP-Z3, TC-Z3, OP-Z4~Z8 | 첫 10분의 결측 시각을 표시하고 다른 컬럼도 같은 시각에 결측인지 확인 | 연속 결측인지 산발 결측인지, 특정 태그만 빠졌는지. 유효 쌍 수와 결측 시작·끝을 기록. 결측을 0으로 바꾸지 않음 |
| 같은 SPHC 후보에서도 표준편차가 다른 이유를 더 볼 수 있는가? | 온도 파일의 LOT 240044·240061·240069·240091 + `lot_grade_review` | 각 LOT 시작부터 **경과시간**을 계산해 TC-Z3와 OP-Z3를 나란히 비교. 첫 10분·마지막 10분·전체 통계를 구분 | 240061의 초반 상승이 다른 SPHC LOT에도 있는지. 서로 다른 날짜 값을 같은 실제 시각처럼 합치지 않고 같은 운전 조건이라고 가정하지 않음 |
| 후보 시작이 강종변경과 겹치는가? | `G-02_조업이벤트.csv`: PLANT_CD, EVT_ID, EVT_DT, EVT_TYPE, BEF_VAL, AFT_VAL | PLANT_CD=CR1, EVT_TYPE=강종변경. 후보 시작·끝 ±10분 조회, 필요 시 ±24시간 별도 표시 | EVT-2024-0122의 DP590→SPHC와 시작 시각 일치 확인. 이벤트 ID·시각·전후 값·후보와 시간 차이 기록. 변경 전 온도가 없어 전환 효과는 계산하지 않음 |
| 초반 상승이 계획수리 다음 관측인가? | `G-01_정기수리캘린더.csv`: PLAN_ID, PLANT_CD, STRT_DT, END_DT, SCOPE | CR1 일정과 LOT 시작 및 후보 구간 대조. 종료일은 날짜 전체 포함으로 처리 | PLAN-2024-02(6/10~6/14) 이후 6/15 관측이라는 맥락. 후보 자체 중첩과 ±24시간 창 중첩을 구분. 실제 재가동이라는 결론은 내리지 않음 |
| 계획 일정 외 실제 개별 작업도 가까이 있었는가? | `G-02_정비이력.csv`: EQP_TAG, WO_NO, DETC_DT, STRT_DT, END_DT, MNT_TYPE, FAIL_MODE, REMARK | EQP_TAG=P1-CR1-CAL01. 발견 시각이 조회 창 안이거나 작업 기간이 겹치는 행 조회 | WO-2024-00972 종료와 후보 시작의 약 7시간 1분 간격 확인. 작업 내용·정비 유형을 설명 자료로 기록. 사후 비고는 예측 입력으로 사용하지 않음 |
| 관련 작업에 교체 부품이 있었는가? | `G-02_부품교체이력.csv`: WO_NO, PART_CD, QTY, CHG_RSN | 위에서 찾은 CAL01 작업의 **WO_NO**로 교체 상세 필터 | 부품 코드·수량·교체 사유 확인. 이번 사건은 대응 교체 0행이며, 기록 없음과 교체가 실제 없었음을 구분 |
| 부품 코드의 의미가 불명확한가? | `G-01_부품마스터.csv`: PART_CD, PART_NM, PART_TYPE, APPLY_EQP_TYPE | 교체 행이 있을 때만 PART_CD로 연결하고 마스터 키 중복 확인 | 부품명·분류 설명에 사용. 현재 사례는 교체 행이 없어 조인하지 않았으며, STD_LIFE_H를 실측 수명·정답 RUL로 사용하지 않음 |
| 기록이 정말 소둔로 CAL01 대상인가? | `G-01_설비마스터.csv`: EQP_TAG, EQP_NM, PARENT_TAG, PLANT_CD, EQP_TYPE | CAL01 태그와 상위 계층 확인 후 관련 기록 필터 | 공장 전체 이벤트와 개별 설비 정비의 범위 차이 기록. 마스터만으로 센서–부품 위치를 확정하지 않음 |

**다음 담당자에게 넘길 표:** `LOT_NO / 강종 후보 / 센서 / 구간 시작·끝 / 선정 이유 / TC·OP 유효수·결측수 / 관련 EVT_ID·WO_NO·PLAN_ID / 관련 기록 시각 / 구간과 시간 차이 / 해석 / 미확정 사항`입니다. 관련 기록이 없으면 ‘해당 조회 범위에 기록 없음’으로 남깁니다. 시간상 가까운 사건을 원인으로 자동 연결하지 않습니다.

**분석 순서:** 같은 파일의 다른 TC·OP·결측 → 같은 후보의 다른 LOT → CR1 강종변경 → CR1 계획수리 → CAL01 개별 정비 → 교체가 있을 때만 부품 상세·마스터. 이렇게 진행하면 온도 자체에서 확인 가능한 현상을 먼저 정리하고 필요한 보조 자료만 사용할 수 있습니다.

### 최종 확인 질문·결과·근거

O는 요청 범위의 검토·기록 완료입니다. 판단 불가를 명확히 기록한 경우도 검토 완료와 실제 효과 입증을 구분합니다.

| 번호 | 확인 질문 | 상태 | 결과·근거 |
| --- | --- | --- | --- |
| 1 | 강종 정의·코드 의미는? | O | 1절 규격·제품 분류와 S1~S5·P1, 로컬 사전 |
| 2 | 강종변경·전후 코드 추출? | O | grade_events, BEF_VAL·AFT_VAL 합집합 |
| 3 | 전체와 CR1 강종 수, 14종 여부? | O | grade_scope: 전체 공장 기록·CR1 모두 8코드; 개별 설비 전수 조사는 아님 |
| 4 | LOT–강종 대응표? | O | lot_grade_review: 관측 시각·이벤트 ID·전후 코드·후보 상태 |
| 5 | 시각 의미·적용 구간·미확정 기록? | O | 시작 일치로 후보 연결; 업무상 적용 시점 미확정 명시 |
| 6 | 특성·용도·열처리 배경·출처? | O | 1절 보완 표. 실제 수치 레시피는 판단 불가 |
| 7 | 문헌 온도와 센서값 직접 비교? | O | 측정 대상·레시피 부재로 직접 비교 불가 명시 |
| 8 | 후보·LOT별 온도 통계? | O | grade_temp_stats·lot_temp_stats의 12센서 6개 통계 |
| 9 | 관측량·운전 조건 한계? | O | lot_summary·lot_gap_stats·missing_by_lot; 미관측 조건 보정 불가 |
| 10 | 변경 전후 TC·OP 변화량? | X: 관측 부재 | event_availability의 양쪽 관측 0건. 가능성 검사 완료, 변화량 산출 제외 |
| 11 | 후보 주변 실제 사건 종류·시각·값? | O | candidate_record·candidate_context·context_counts; 후보 1구간 수행 |
| 12 | 하단 결과·근거·그래프·한계·인계? | O | 본 표와 실제 대조 결과, [4절 그래프 3개](#plots), 아래 후속 과제 |

### 남은 보완 작업 — 현재 파일만 사용

1. **우선: 초반 OP 결측의 시간 위치 확인.** missing_by_lot와 해당 시계열로 결측의 연속성·다른 센서 동반 결측을 확인합니다. 원인을 단정하지 않습니다.
2. **구간 선택 민감도 확인.** 첫·마지막 10분을 5분·15분으로 바꿔 수준 차이가 유지되는지 봅니다. 10분은 운영 기준이 아닌 탐색 창입니다.
3. **기존 반복 LOT 비교.** SPHC 4 LOT를 실제 시작 이후 경과시간으로 나란히 보고 비슷한 초반 추세가 반복되는지 확인합니다. 경과시간 정렬이 같은 운전 상태를 보장하지는 않습니다.
4. **다른 존·출력의 동반 변화 검토.** 같은 시각 유효값만 사용하고 존별 추세·지연을 분리합니다. 공백을 넘어 보간하거나 행 이동을 초로 간주하지 않습니다.
5. **다른 후보에 동일 사건 대조 적용.** 후보 구간·선정 이유를 먼저 기록하고 candidate_context와 같은 조회를 반복합니다. 미관측은 정상으로 처리하지 않습니다.

현재 확인된 결과는 ‘관측 수준 차이·초반 추세·출력 결측·주변 사건’입니다. 설정값·속도·두께가 없는 상태에서 강종 단독 효과·정상화·수리 효과·고장 인과관계를 추가 결론으로 만들지 않습니다.

[↑ 목차](#toc)
